# Avaliação por Partidas Reais — CNN vs Minimax e vs Oráculo

Este notebook foi isolado para utilizar o ambiente `.venv_tf` (TensorFlow 2.21+), que possui compatibilidade com o modelo `.tflite` exportado no Google Colab (opcodes mais recentes).

**Instrução Importante:** Certifique-se de selecionar o Kernel `.venv_tf` (ou Python do `.venv_tf`) no VS Code antes de rodar a célula abaixo.

In [6]:
import sys, os, io
from datetime import datetime
from pathlib import Path

sys.path.insert(0, os.path.abspath('../..'))

from tqdm.auto import tqdm

from gerador_dados.jogo_pontinhos.avaliador_partidas_pontinhos import (
    avaliar_paralelo, imprimir_relatorio,
    todos_labels_canonicos
)

# =========================================================================
# CONFIGURAÇÃO
# =========================================================================
# >>> EDITE: tag do experimento — DEVE bater com a tag usada no treino <<<
EXPERIMENTO     = 'boxnetv4_oraculo_exato_refinamento2_8p3M'
TAMANHO         = 'pequeno'
N_PARTIDAS      = 200    # total por profundidade (100 CNN 1ª a jogar + 100 CNN 2ª a jogar)
PROFUNDIDADES   = [1, 3, 5, 6]
TIMER_MS        = 0      # 0 = sem limite | ex: 5 = Minimax limitado a 5ms/jogada
MAX_WORKERS     = 14

# Canais usados no treinamento deste modelo (para documentação do relatório)
# CANAIS_MODELO = [
#     'aresta_topo', 'aresta_base', 'aresta_esquerda', 'aresta_direita', 'caixa_fechada',
# ]

CANAIS_MODELO = [
    'aresta_topo', 'aresta_base', 'aresta_esquerda', 'aresta_direita',
    'caixa_fechada', 'eh_grau3', 'eh_grau2', 'em_cadeia_curta',
    'em_cadeia_longa', 'em_loop', 'em_cadeia_aberta_uma_ponta',
    'paridade_cadeia_longa_impar'
]

# Nº de canais e nomes de arquivo derivados da tag (12 canais; use 5 p/ ablação R1).
_NC = len(CANAIS_MODELO)
CAMINHO_MODELO  = f'../../modelos/pontinhos_pequeno_cnn_{_NC}canais_{EXPERIMENTO}.tflite'
RELATORIO       = f'pontinhos_pequeno_cnn_{_NC}canais_{EXPERIMENTO}_avaliacao'

SALVAR_CAIXAS_PERDIDAS = False
BASE_VIS = Path('../../visualizacoes/avaliacao_partidas')
EXEC_ID  = f"{Path(CAMINHO_MODELO).stem}__{datetime.now().strftime('%Y%m%d_%H%M%S')}"
EXEC_DIR = BASE_VIS / EXEC_ID

# =========================================================================
# CAPTURA DE RELATÓRIO
# =========================================================================
_report_buf = io.StringIO()

def rprint(*args, sep=' ', end='\n'):
    print(*args, sep=sep, end=end)
    msg = sep.join(str(a) for a in args) + end
    _report_buf.write(msg)

def _md_h(level, text):
    rprint('\n' + '#' * level + ' ' + text + '\n')

def _md_sep():
    rprint('\n---\n')

def _md_table_rows(rows, index_col=None):
    """Tabela markdown a partir de lista de dicts — sem pandas."""
    if not rows:
        return
    all_cols = list(rows[0].keys())
    if index_col and index_col in all_cols:
        all_cols.remove(index_col)
        header = [index_col] + all_cols
    else:
        header = all_cols
        index_col = None

    data = []
    for row in rows:
        data.append([str(row.get(c, '')) for c in header])

    widths = [len(h) for h in header]
    for row in data:
        for i, cell in enumerate(row):
            widths[i] = max(widths[i], len(cell))

    fmt = lambda cells: '| ' + ' | '.join(c.ljust(widths[i]) for i, c in enumerate(cells)) + ' |'
    rprint(fmt(header))
    rprint('| ' + ' | '.join('-' * w for w in widths) + ' |')
    for row in data:
        rprint(fmt(row))
    rprint()

# =========================================================================
# EXECUÇÃO
# =========================================================================
labels = todos_labels_canonicos(4, 3)

# Cabeçalho do relatório
_md_h(1, 'Relatório de Avaliação — CNN vs Minimax e Oráculo')
rprint('| Parâmetro | Valor |')
rprint('|-----------|-------|')
rprint(f'| Data | {datetime.now().strftime("%Y-%m-%d %H:%M")} |')
rprint(f'| Modelo | `{Path(CAMINHO_MODELO).name}` |')
rprint(f'| Canais ({len(CANAIS_MODELO)}) | {", ".join(CANAIS_MODELO)} |')
rprint(f'| Partidas por profundidade | {N_PARTIDAS} |')
rprint(f'| Profundidades | {PROFUNDIDADES} |')
timer_desc = f'{TIMER_MS}ms/jogada (iterative deepening)' if TIMER_MS > 0 else 'sem limite'
rprint(f'| Timer Minimax | {timer_desc} |')
rprint()
_md_sep()

stats_list = []
if __name__ == '__main__':
    if TIMER_MS > 0:
        print(f'Timer ativo: Minimax limitado a {TIMER_MS}ms por jogada')
    if SALVAR_CAIXAS_PERDIDAS:
        print(f'Salvando visualizações de caixas perdidas em: {EXEC_DIR}')

    for prof in PROFUNDIDADES:
        nome = f'Minimax(p<={prof}, {TIMER_MS}ms)' if TIMER_MS > 0 else f'Minimax(p={prof})'

        with tqdm(total=N_PARTIDAS, desc=f'{nome:25s}', unit='partida', leave=True) as pbar:
            c = [0, 0, 0, 0, 0]

            def _cb(completed, total, result, _pbar=pbar, _c=c):
                if result['vencedor'] == 1:   _c[0] += 1
                elif result['vencedor'] == 0:  _c[1] += 1
                else:                          _c[2] += 1
                _c[3] += result.get('opp_perdidas_a1', 0)
                _c[4] += result.get('opp_total_a1', 0)
                _pbar.update(completed - _pbar.n)
                postfix = {'V': _c[0], 'E': _c[1], 'D': _c[2]}
                if _c[4] > 0:
                    postfix['?caixa'] = f"{_c[3]}/{_c[4]}"
                _pbar.set_postfix(postfix)

            saida_misses = (EXEC_DIR / f'minimax_p{prof}') if SALVAR_CAIXAS_PERDIDAS else None

            s = avaliar_paralelo(
                CAMINHO_MODELO, labels, prof, nome, TAMANHO,
                N_PARTIDAS, TIMER_MS, MAX_WORKERS,
                progress_callback=_cb,
                salvar_caixas_perdidas_em=saida_misses,
            )
        stats_list.append(s)

    imprimir_relatorio(stats_list)


# Relatório de Avaliação — CNN vs Minimax e Oráculo

| Parâmetro | Valor |
|-----------|-------|
| Data | 2026-06-01 17:27 |
| Modelo | `pontinhos_pequeno_cnn_12canais_boxnetv4_oraculo_exato_refinamento2_8p3M.tflite` |
| Canais (12) | aresta_topo, aresta_base, aresta_esquerda, aresta_direita, caixa_fechada, eh_grau3, eh_grau2, em_cadeia_curta, em_cadeia_longa, em_loop, em_cadeia_aberta_uma_ponta, paridade_cadeia_longa_impar |
| Partidas por profundidade | 200 |
| Profundidades | [1, 3, 5, 6] |
| Timer Minimax | sem limite |


---



Minimax(p=6)             : 100%|██████████| 200/200 [16:55<00:00,  5.08s/partida, V=192, E=8, D=0, ?caixa=89/1839]


AVALIAÇÃO POR PARTIDAS REAIS — CNN vs Minimax

  Adversário: Minimax(p=1)  (200 partidas)
  Vitórias CNN          200  (100.0%)
  Empates                 0  (  0.0%)
  Derrotas CNN            0  (  0.0%)
  Tempo médio CNN:  8.87 ms/jogada
  Tempo médio Minimax(p=1): 0.3 ms/jogada
  CNN é 0× mais rápida
  Caixas deixadas p/ Minimax:   76 / 2030 oportunidades (3.7%)

  Adversário: Minimax(p=3)  (200 partidas)
  Vitórias CNN          194  ( 97.0%)
  Empates                 6  (  3.0%)
  Derrotas CNN            0  (  0.0%)
  Tempo médio CNN:  2.17 ms/jogada
  Tempo médio Minimax(p=3): 88.7 ms/jogada
  CNN é 41× mais rápida
  Caixas deixadas p/ Minimax:   86 / 1887 oportunidades (4.6%)

  Adversário: Minimax(p=5)  (200 partidas)
  Vitórias CNN          192  ( 96.0%)
  Empates                 7  (  3.5%)
  Derrotas CNN            1  (  0.5%)
  Tempo médio CNN:  2.00 ms/jogada
  Tempo médio Minimax(p=5): 1689.6 ms/jogada
  CNN é 844× mais rápida
  Caixas deixadas p/ Minimax:   87 / 1857 opor

In [7]:
# =========================================================================
# CNN vs ORÁCULO (jogo perfeito = Minimax p=31) — métrica de teto, quebrada por QUEM ABRE
# Roda LOCALMENTE (.venv_tf) com a tablebase exata do 4x3 como adversário.
# Protocolo: 1a jogada ALEATÓRIA (quebra a repeticao deterministica); alterna
# quem ABRE. O ORACULO = Minimax de profundidade total (p=31), o teto de forca (mais forte que qualquer Minimax p<=19). Nao se vence o jogo perfeito (4x3 e empate sob jogo otimo) — as
# vitorias da CNN vem de aberturas que ja a favorecem; por isso separamos os
# resultados por QUEM ABRIU (CNN abre x Oraculo abre).
# =========================================================================
import random
from gerador_dados.jogo_pontinhos.oraculo_tablebase_pontinhos import (
    construir_mapeamento, carregar, scores_de_todas_jogadas_exato, matriz_para_bitmask)
from gerador_dados.jogo_pontinhos.avaliador_partidas_pontinhos import _cnn_agent_fn
from analise.jogo_pontinhos.diagnostico_derrotas_cnn_pequeno_referencia.arena_pontinhos import (
    jogar_partida_instrumentada)

TABLEBASE_ORACULO = '../../dados/oraculo_pontinhos/tablebase_pequeno_4x3.npy'
N_ORACULO = 200   # total; metade CNN abre, metade Oraculo abre (alternado)

_val_ora = carregar(TABLEBASE_ORACULO, mmap=True)
_lbls_ora, _erc, _ne, _ebm, _ebc, _ = construir_mapeamento(4, 3)

def _agente_oraculo(estado):
    # Jogo PERFEITO: argmax exato do oraculo; desempate aleatorio entre otimos.
    s = matriz_para_bitmask(estado.matriz, 4, 3, _erc)
    sc = scores_de_todas_jogadas_exato(_val_ora, s, _ne, _ebm, _ebc)
    qmax = max(sc.values())
    return _lbls_ora[random.choice([e for e, q in sc.items() if q == qmax])]

_cnn_ora = _cnn_agent_fn(CAMINHO_MODELO, labels)
_res = {'CNN': [], 'Oraculo': []}   # resultados (perspectiva CNN) por quem ABRE
for _seed in tqdm(range(N_ORACULO), desc='CNN vs Oráculo', unit='partida'):
    _cnn_abre = (_seed % 2 == 0)    # quem e jogador 1 = quem abre (faz a 1a jogada)
    _r = jogar_partida_instrumentada(
        _cnn_ora, _agente_oraculo, ref_eh_jogador1=_cnn_abre,
        tamanho=TAMANHO, seed=_seed, lances_abertura_aleatorios=1)
    _res['CNN' if _cnn_abre else 'Oraculo'].append(_r.resultado_ref)

def _wdl(lst):
    return lst.count('vitoria'), lst.count('empate'), lst.count('derrota')

_todos = _res['CNN'] + _res['Oraculo']
_md_h(2, 'CNN vs Oráculo (jogo perfeito = Minimax p=31)')
rprint('_Protocolo: 1a jogada aleatoria; alterna quem ABRE. O ORACULO = Minimax de profundidade total (p=31), o teto de forca (mais forte que qualquer Minimax p<=19). Nao se vence o jogo perfeito '
       '(4x3 e empate sob jogo otimo) — vitorias da CNN vem de aberturas favoraveis. '
       'Por isso a quebra por QUEM ABRE._\n')
rprint('| Quem abre | Partidas | Vitorias CNN | Empates | Derrotas CNN |')
rprint('|-----------|----------|--------------|---------|--------------|')
for _nome, _lst in [('TOTAL', _todos), ('CNN abre', _res['CNN']), ('Oraculo abre', _res['Oraculo'])]:
    _n = len(_lst)
    if _n == 0:
        continue
    _vv, _ee, _dd = _wdl(_lst)
    rprint(f'| {_nome} | {_n} | {_vv} ({_vv/_n*100:.1f}%) | {_ee} ({_ee/_n*100:.1f}%) | {_dd} ({_dd/_n*100:.1f}%) |')
rprint()
_md_sep()
print(f'CNN vs Oraculo -> CNN abre: {_wdl(_res["CNN"])} | Oraculo abre: {_wdl(_res["Oraculo"])} (V,E,D)')


CNN vs Oráculo: 100%|██████████| 200/200 [00:04<00:00, 44.76partida/s]


## CNN vs Oráculo (jogo perfeito = Minimax p=31)

_Protocolo: 1a jogada aleatoria; alterna quem ABRE. O ORACULO = Minimax de profundidade total (p=31), o teto de forca (mais forte que qualquer Minimax p<=19). Nao se vence o jogo perfeito (4x3 e empate sob jogo otimo) — vitorias da CNN vem de aberturas favoraveis. Por isso a quebra por QUEM ABRE._

| Quem abre | Partidas | Vitorias CNN | Empates | Derrotas CNN |
|-----------|----------|--------------|---------|--------------|
| TOTAL | 200 | 80 (40.0%) | 31 (15.5%) | 89 (44.5%) |
| CNN abre | 100 | 0 (0.0%) | 12 (12.0%) | 88 (88.0%) |
| Oraculo abre | 100 | 80 (80.0%) | 19 (19.0%) | 1 (1.0%) |


---

CNN vs Oraculo -> CNN abre: (0, 12, 88) | Oraculo abre: (80, 19, 1) (V,E,D)


In [8]:
# =========================================================================
# RELATÓRIO MARKDOWN — Resultados da Avaliação
# Executar após a célula principal. Salva .md na pasta do projeto.
# =========================================================================
_md_h(2, 'Resultados por Adversário')

# Tabela resumo
rows = []
for s in stats_list:
    fator = f"{s['fator_velocidade']:.0f}×"
    opp = '—'
    if s.get('opp_total_cnn', 0) > 0:
        opp_p = f"{s['opp_perdidas_cnn'] / s['opp_total_cnn'] * 100:.1f}%"
        opp = f"{s['opp_perdidas_cnn']}/{s['opp_total_cnn']} ({opp_p})"
    rows.append({
        'Adversário':      s['adversario'],
        'Partidas':        str(s['partidas']),
        'Vitórias CNN':    f"{s['vitorias_cnn']} ({s['pct_vitorias']:.1f}%)",
        'Empates':         f"{s['empates']} ({s['pct_empates']:.1f}%)",
        'Derrotas CNN':    f"{s['derrotas_cnn']} ({s['pct_derrotas']:.1f}%)",
        'T. CNN (ms)':     f"{s['tempo_cnn_ms']:.2f}",
        'T. MM (ms)':      f"{s['tempo_mm_ms']:.1f}",
        'CNN mais rápida': fator,
        'Caixas cedidas':  opp,
    })

_md_table_rows(rows, index_col='Adversário')

# Detalhes por adversário
_md_h(2, 'Detalhes por Adversário')
for s in stats_list:
    _md_h(3, s['adversario'])
    rprint('| Métrica | Valor |')
    rprint('|---------|-------|')
    rprint(f'| Vitórias CNN | {s["vitorias_cnn"]} ({s["pct_vitorias"]:.1f}%) |')
    rprint(f'| Empates | {s["empates"]} ({s["pct_empates"]:.1f}%) |')
    rprint(f'| Derrotas CNN | {s["derrotas_cnn"]} ({s["pct_derrotas"]:.1f}%) |')
    rprint(f'| Tempo médio CNN | {s["tempo_cnn_ms"]:.2f} ms/jogada |')
    rprint(f'| Tempo médio Minimax | {s["tempo_mm_ms"]:.1f} ms/jogada |')
    rprint(f'| CNN é mais rápida | {s["fator_velocidade"]:.0f}× |')
    if s.get('opp_total_cnn', 0) > 0:
        opp_pct = s['opp_perdidas_cnn'] / s['opp_total_cnn'] * 100
        rprint(f'| Caixas cedidas ao Minimax | {s["opp_perdidas_cnn"]}/{s["opp_total_cnn"]} ({opp_pct:.1f}%) |')
    if 'prof_media_mm' in s:
        rprint(f'| Profundidade média alcançada (MM) | {s["prof_media_mm"]:.1f} |')
    rprint()

_md_sep()

# Salvar relatório
ts = datetime.now().strftime('%Y%m%d_%H%M%S')
nome_md = f"{RELATORIO}_{ts}.md"
caminho_md = Path('../../resultados/jogo_pontinhos') / nome_md
caminho_md.parent.mkdir(parents=True, exist_ok=True)

with open(caminho_md, 'w', encoding='utf-8') as f:
    f.write(_report_buf.getvalue())
print(f'Relatório salvo: {caminho_md} ({caminho_md.stat().st_size / 1024:.1f} KB)')


## Resultados por Adversário

| Adversário   | Partidas | Vitórias CNN | Empates  | Derrotas CNN | T. CNN (ms) | T. MM (ms) | CNN mais rápida | Caixas cedidas |
| ------------ | -------- | ------------ | -------- | ------------ | ----------- | ---------- | --------------- | -------------- |
| Minimax(p=1) | 200      | 200 (100.0%) | 0 (0.0%) | 0 (0.0%)     | 8.87        | 0.3        | 0×              | 76/2030 (3.7%) |
| Minimax(p=3) | 200      | 194 (97.0%)  | 6 (3.0%) | 0 (0.0%)     | 2.17        | 88.7       | 41×             | 86/1887 (4.6%) |
| Minimax(p=5) | 200      | 192 (96.0%)  | 7 (3.5%) | 1 (0.5%)     | 2.00        | 1689.6     | 844×            | 87/1857 (4.7%) |
| Minimax(p=6) | 200      | 192 (96.0%)  | 8 (4.0%) | 0 (0.0%)     | 2.00        | 5402.3     | 2703×           | 89/1839 (4.8%) |


## Detalhes por Adversário


### Minimax(p=1)

| Métrica | Valor |
|---------|-------|
| Vitórias CNN | 200 (100.0%) |
| Empates | 0 (0.0%) |
| Derrotas CNN | 0 (0.0%) |
| Tempo méd

## Resultado Profundidade 6

In [9]:
# Avaliando contra Minimax(p=1) em paralelo (multi-core)...
# Avaliando contra Minimax(p=3) em paralelo (multi-core)...
# Avaliando contra Minimax(p=5) em paralelo (multi-core)...
# Avaliando contra Minimax(p=6) em paralelo (multi-core)...
# 
# ========================================================================
# AVALIAÇÃO POR PARTIDAS REAIS — CNN vs Minimax
# **CNN TREINADA COM TABULEIROS GERADOS DE FORMA ALEATÓRIA**
# ========================================================================
# 
#   Adversário: Minimax(p=1)  (200 partidas)
#   Vitórias CNN          192  ( 96.0%)
#   Empates                 4  (  2.0%)
#   Derrotas CNN            4  (  2.0%)
#   Tempo médio CNN:  0.06 ms/jogada
#   Tempo médio Minimax(p=1): 0.2 ms/jogada
#   CNN é 3× mais rápida
# 
#   Adversário: Minimax(p=3)  (200 partidas)
#   Vitórias CNN          105  ( 52.5%)
#   Empates                26  ( 13.0%)
#   Derrotas CNN           69  ( 34.5%)
#   Tempo médio CNN:  0.10 ms/jogada
#   Tempo médio Minimax(p=3): 82.4 ms/jogada
#   CNN é 847× mais rápida
# 
#   Adversário: Minimax(p=5)  (200 partidas)
#   Vitórias CNN           44  ( 22.0%)
#   Empates                13  (  6.5%)
#   Derrotas CNN          143  ( 71.5%)
#   Tempo médio CNN:  0.13 ms/jogada
#   Tempo médio Minimax(p=5): 1639.1 ms/jogada
#   CNN é 12519× mais rápida
# 
#   Adversário: Minimax(p=6)  (200 partidas)
#   Vitórias CNN            3  (  1.5%)
#   Empates                10  (  5.0%)
#   Derrotas CNN          187  ( 93.5%)
#   Tempo médio CNN:  0.15 ms/jogada
#   Tempo médio Minimax(p=6): 5107.0 ms/jogada
#   CNN é 34794× mais rápida
# 
# ========================================================================

## Resultado Profundidade 7

In [10]:
# Avaliando contra Minimax(p=1) em paralelo (multi-core)...
#   Progresso: [200/200] 100.0% concluído...
# Avaliando contra Minimax(p=3) em paralelo (multi-core)...
#   Progresso: [200/200] 100.0% concluído...
# Avaliando contra Minimax(p=5) em paralelo (multi-core)...
#   Progresso: [200/200] 100.0% concluído...
# Avaliando contra Minimax(p=6) em paralelo (multi-core)...
#   Progresso: [200/200] 100.0% concluído...
# 
# ========================================================================
# AVALIAÇÃO POR PARTIDAS REAIS — CNN vs Minimax
# ========================================================================
# 
#   Adversário: Minimax(p=1)  (200 partidas)
#   Vitórias CNN          184  ( 92.0%)
#   Empates                12  (  6.0%)
#   Derrotas CNN            4  (  2.0%)
#   Tempo médio CNN:  0.09 ms/jogada
#   Tempo médio Minimax(p=1): 0.2 ms/jogada
#   CNN é 2× mais rápida
# 
#   Adversário: Minimax(p=3)  (200 partidas)
#   Vitórias CNN          120  ( 60.0%)
#   Empates                54  ( 27.0%)
#   Derrotas CNN           26  ( 13.0%)
#   Tempo médio CNN:  0.11 ms/jogada
#   Tempo médio Minimax(p=3): 80.6 ms/jogada
#   CNN é 711× mais rápida
# 
#   Adversário: Minimax(p=5)  (200 partidas)
#   Vitórias CNN           89  ( 44.5%)
#   Empates                59  ( 29.5%)
#   Derrotas CNN           52  ( 26.0%)
#   Tempo médio CNN:  0.13 ms/jogada
#   Tempo médio Minimax(p=5): 1627.1 ms/jogada
#   CNN é 12419× mais rápida
# 
#   Adversário: Minimax(p=6)  (200 partidas)
#   Vitórias CNN           80  ( 40.0%)
#   Empates                46  ( 23.0%)
#   Derrotas CNN           74  ( 37.0%)
#   Tempo médio CNN:  0.14 ms/jogada
#   Tempo médio Minimax(p=6): 5820.3 ms/jogada
#   CNN é 41782× mais rápida
# 
# ========================================================================